<!--nav--> [🗺 Learning path](README.md) · **33/48** · ◀ [Serving LoRA Adapters at Scale](./MultiLoRA_Serving_At_Scale.ipynb) · [GPU Architecture & CUDA Kernels](./GPU_Architecture_And_CUDA_Kernels.ipynb) ▶

# The Hardware Roofline: NVIDIA vs AMD, from First Principles

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sugeerth/gpu-training-notebooks/blob/main/Hardware_Roofline_NVIDIA_vs_AMD.ipynb)

Every serving notebook so far assumed one GPU family. Production doesn't. You will be handed
whatever your cloud has capacity for — **T4, L4, A100, H100, H200, B200, MI210, MI250X, MI300X,
MI325X** — and the *optimal serving configuration is different on each one*.

This notebook gives you the model that explains all of them at once: the **roofline**. From two
numbers per GPU (memory bandwidth and peak FLOPS) you can predict decode speed, prefill speed, the
batch size where your bottleneck flips, and whether quantization will help you *at all* on that
specific card.

| Part | What you'll learn |
|---|---|
| **1** | The architectural vocabulary, translated: SM↔CU, warp↔wavefront, Tensor↔Matrix cores |
| **2** | The spec table that matters — and the two numbers that predict everything |
| **3** | **The roofline model**, derived for LLM inference (not the generic textbook version) |
| **4** | **The flip point**: the exact batch size where decode stops being memory-bound |
| **5** | Why quantization moves that point — and can *hurt* on some cards |
| **6** | Capacity as an architecture decision: how MI300X's 192 GB changes your parallelism plan |
| **7** | A **portable microbenchmark** that runs on CUDA *or* ROCm and measures your real roofline |

**Runs on:** any CPU for Parts 1–6. Part 7 detects and uses whatever GPU you have (NVIDIA *or* AMD).

> ⚠️ **On the numbers.** Peak specs below are vendor-published figures for **dense** (non-sparse)
> math, gathered at the time of writing. Vendors change datasheets, and "peak" is a number you will
> never actually hit. **Verify against current datasheets before making purchasing decisions** — and
> treat the *method* here as the durable part, not the constants. Part 7 measures your real numbers,
> which are typically 60–85% of peak.

In [ ]:
import math, json, uuid
from IPython.display import HTML, display

D3_URL = "https://cdn.jsdelivr.net/npm/d3@7/dist/d3.min.js"
def show_d3(js, data=None, height=420):
    div = f"viz_{uuid.uuid4().hex[:10]}"
    html = f'''
<div id="{div}" style="width:100%;max-width:920px;font-family:system-ui,sans-serif"></div>
<script>
(function() {{
  function run() {{
    const d3 = window.d3, root = d3.select("#{div}"), data = {json.dumps(data)};
    const W = (document.getElementById("{div}").clientWidth || 880), H = {height};
    try {{ {js} }} catch (e) {{ root.append("pre").style("color","crimson").text("viz error: " + e); }}
  }}
  if (window.d3) run();
  else {{ const s = document.createElement("script"); s.src = "{D3_URL}"; s.onload = run;
          s.onerror = () => document.getElementById("{div}").textContent = "Could not load D3.";
          document.head.appendChild(s); }}
}})();
</script>'''
    display(HTML(html))
print("ready")

## Part 1 · The same machine, two vocabularies

AMD and NVIDIA build remarkably similar things and name almost nothing the same way. Half of
"AMD is different" is really "AMD is spelled differently":

| Concept | NVIDIA | AMD | Actually different? |
|---|---|---|---|
| Core cluster | **SM** (Streaming Multiprocessor) | **CU** (Compute Unit) | no — same role |
| SIMD group | **warp** = 32 lanes | **wavefront** = 64 lanes | **yes** — affects kernel tuning, occupancy, and reductions |
| Matrix engine | **Tensor Core** | **Matrix Core** | no — same job, different instruction names (`mma` vs `mfma`) |
| Fast scratchpad | **shared memory** | **LDS** (Local Data Share) | no |
| Programming API | **CUDA** | **HIP** | ~95% source-compatible; `hipify` mechanically translates most code |
| BLAS | cuBLAS / cuBLASLt | **rocBLAS / hipBLASLt** | no — same role, different tuning maturity |
| Templates library | **CUTLASS** | **Composable Kernel (CK)** | no |
| Collectives | **NCCL** | **RCCL** | no — RCCL is a port of NCCL's API |
| Scale-up fabric | **NVLink / NVSwitch** | **Infinity Fabric / xGMI** | topology details differ; role identical |
| Profiler | Nsight / `nvidia-smi` | `rocprof` / `rocm-smi` | no |
| Graph capture | **CUDA Graphs** | **HIP Graphs** | no |

**The one that genuinely bites you:** *wavefront 64 vs warp 32*. A kernel hand-tuned for 32-lane
warps (tile sizes, reduction trees, occupancy math) is often merely *correct* rather than *fast*
after a mechanical port. That's why AMD ships its own kernel libraries (CK, and the
inference-focused **AITER**) rather than relying on translated CUDA — and why "does it run?" and
"does it run *fast*?" are different questions on a new vendor.

The second thing that bites: **ecosystem maturity per feature**, not per vendor. [Portable Kernels & Precision](./Portable_Kernels_Precision_Matrix.ipynb) maps
that in detail.

## Part 2 · The two numbers that predict everything

For LLM *inference* you can ignore almost every spec except:

1. **HBM bandwidth (TB/s)** — decode speed lives here ([Serving Fundamentals](./Serving_Fundamentals_KV_Cache_Batching.ipynb): decode is memory-bound).
2. **Peak dense FLOPS at your dtype** — prefill speed lives here.

Plus one more that decides your *topology* rather than your speed: **VRAM capacity**.

In [ ]:
# Vendor-published PEAK figures, dense (no sparsity), gathered at the time of writing.
# VERIFY against current datasheets. The method matters more than these constants.
GPUS = [
  # name              vendor    arch        GB    BW TB/s  fp16 TF  fp8 TF   $/hr   notes
  ("T4",              "NVIDIA", "Turing",     16,   0.32,     65,    None,   0.35, "no bf16, no FP8"),
  ("L4",              "NVIDIA", "Ada",        24,   0.30,    121,     242,   0.70, "FP8 capable, low BW"),
  ("A10G",            "NVIDIA", "Ampere",     24,   0.60,    125,    None,   1.00, "no FP8"),
  ("A100 40GB",       "NVIDIA", "Ampere",     40,   1.56,    312,    None,   1.80, "no FP8; Marlin int4"),
  ("A100 80GB",       "NVIDIA", "Ampere",     80,   2.04,    312,    None,   2.20, "no FP8"),
  ("H100 SXM",        "NVIDIA", "Hopper",     80,   3.35,    990,    1979,   3.50, "FP8 native"),
  ("H200 SXM",        "NVIDIA", "Hopper",    141,   4.80,    990,    1979,   4.50, "H100 compute, more HBM"),
  ("B200",            "NVIDIA", "Blackwell", 192,   8.00,   2250,    4500,   7.00, "FP4/FP6; approximate"),
  ("MI210",           "AMD",    "CDNA2",      64,   1.60,    181,    None,   1.20, "no FP8"),
  ("MI250X",          "AMD",    "CDNA2",     128,   3.20,    383,    None,   2.20, "2 dies, exposed as 2 GPUs"),
  ("MI300X",          "AMD",    "CDNA3",     192,   5.30,   1307,    2615,   3.50, "FP8 native, huge VRAM"),
  ("MI325X",          "AMD",    "CDNA3",     256,   6.00,   1307,    2615,   4.20, "MI300X compute, more HBM"),
  ("MI355X",          "AMD",    "CDNA4",     288,   8.00,   2300,    4600,   7.00, "FP4/FP6; approximate"),
]
COLS = ("name", "vendor", "arch", "vram_gb", "bw_tbs", "fp16_tf", "fp8_tf", "usd_hr", "notes")
GPU = [dict(zip(COLS, g)) for g in GPUS]

print(f"{'GPU':<14}{'vendor':<8}{'arch':<11}{'VRAM':>7}{'BW TB/s':>9}{'fp16 TF':>9}"
      f"{'fp8 TF':>8}{'$/hr':>7}   ridge point*")
print("-" * 96)
for g in GPU:
    # ridge point = peak FLOPS / bandwidth = arithmetic intensity where compute takes over
    ridge = g["fp16_tf"] * 1e12 / (g["bw_tbs"] * 1e12)
    g["ridge_fp16"] = ridge
    print(f"{g['name']:<14}{g['vendor']:<8}{g['arch']:<11}{g['vram_gb']:>5}GB{g['bw_tbs']:>9.2f}"
          f"{g['fp16_tf']:>9.0f}{(g['fp8_tf'] or 0):>8.0f}{g['usd_hr']:>7.2f}   {ridge:>6.0f} FLOP/byte")

print("\n* The RIDGE POINT is the star of this notebook. It is the arithmetic intensity")
print("  (FLOPs performed per byte moved) at which a kernel stops being memory-bound and")
print("  starts being compute-bound. Everything below is a consequence of it.")

## Part 3 · The roofline, derived for LLM inference

The generic roofline says: achievable performance = `min(peak_FLOPS, arithmetic_intensity × bandwidth)`.
Let's make that concrete for the two phases of inference ([Serving Fundamentals](./Serving_Fundamentals_KV_Cache_Batching.ipynb)).

### Decode

Per generated token, per request, a model with **N parameters**:

- **compute**: ~`2N` FLOPs (one multiply-add per weight)
- **memory**: `N × bytes_per_weight` — but this read is **shared across the whole batch**

So for batch size `B`:

$$\text{AI}_{decode} = \frac{2NB}{N \cdot \text{bytes}} = \frac{2B}{\text{bytes}}$$

**The batch size vanishes from the memory term but not the compute term** — that is the entire
reason batching works. And it gives us the flip point directly. Setting `AI = ridge`:

$$B^* = \frac{\text{ridge} \times \text{bytes}}{2}$$

For fp16 (2 bytes), `B* = ridge`. Let's compute it for every card:

In [ ]:
def flip_point(g, bytes_per_weight=2.0):
    '''Batch size at which decode stops being memory-bound on this GPU.'''
    return g["ridge_fp16"] * bytes_per_weight / 2

def decode_tps_ceiling(g, params_b, bytes_per_weight=2.0, efficiency=0.75):
    '''Memory-bound ceiling: tokens/s = bandwidth / bytes_read_per_token.'''
    model_bytes = params_b * 1e9 * bytes_per_weight
    return g["bw_tbs"] * 1e12 * efficiency / model_bytes

print("Decode ceiling for an 8B model in fp16 (75% of peak bandwidth achieved),")
print("and the batch size where each GPU flips from memory-bound to compute-bound:\n")
print(f"{'GPU':<14}{'tok/s (B=1)':>13}{'flip point B*':>15}{'fp16':>7}{'int4':>7}   what it means")
print("-" * 92)
for g in GPU:
    tps = decode_tps_ceiling(g, 8.0)
    b16, b4 = flip_point(g, 2.0), flip_point(g, 0.5)
    note = ("batch beyond this buys latency, not throughput" if b16 > 150
            else "flips to compute-bound early - watch kernel quality")
    print(f"{g['name']:<14}{tps:>13.0f}{'':>15}{b16:>7.0f}{b4:>7.0f}   {note}")

print("\nRead the int4 column carefully - this is the insight most people miss:")
print("quantizing to 4 bits divides the flip point by 4. You reach the compute roof at")
print("a batch 4x smaller, because you made the memory side cheap without touching compute.")

### Prefill

Prefill processes `T` prompt tokens in parallel, so:

- **compute**: `2NT` FLOPs
- **memory**: still just `N × bytes` (the weights, read once for the whole chunk)

$$\text{AI}_{prefill} = \frac{2T}{\text{bytes}}$$

With `T` in the hundreds or thousands, prefill sails past every ridge point on the market — it is
**always compute-bound**. That's why:

- prefill speed tracks **FLOPS**, decode speed tracks **bandwidth**;
- quantizing weights barely speeds up prefill (you didn't add FLOPS) but clearly speeds up decode;
- the two phases want different hardware, which is the whole argument for disaggregation ([Distributed & Multi-Replica Serving](./Distributed_MultiReplica_Serving.ipynb)).

Let's put both phases and every GPU on one chart:

In [ ]:
# Build roofline curves: performance vs arithmetic intensity, per GPU.
roof = []
for g in GPU:
    peak = g["fp16_tf"] * 1e12
    bw = g["bw_tbs"] * 1e12
    pts = []
    ai = 0.25
    while ai <= 4096:
        pts.append({"ai": ai, "perf": min(peak, ai * bw) / 1e12})
        ai *= 1.35
    roof.append({"name": g["name"], "vendor": g["vendor"], "peak_tf": g["fp16_tf"],
                 "bw": g["bw_tbs"], "ridge": g["ridge_fp16"], "points": pts})

# Workload markers: where real LLM inference sits on the AI axis (fp16 weights)
workloads = [
    {"label": "decode B=1",    "ai": 2 * 1 / 2},
    {"label": "decode B=8",    "ai": 2 * 8 / 2},
    {"label": "decode B=64",   "ai": 2 * 64 / 2},
    {"label": "decode B=256",  "ai": 2 * 256 / 2},
    {"label": "prefill T=2048","ai": 2 * 2048 / 2},
]
print("workload arithmetic intensities (FLOP/byte, fp16 weights):")
for w in workloads:
    print(f"  {w['label']:<16}{w['ai']:>8.0f}")
print("\nCompare each to the ridge points in Part 2: anything to the LEFT of a GPU's ridge")
print("is memory-bound on that GPU; anything to the RIGHT is compute-bound.")

In [ ]:
JS = r'''
const M = {top: 20, right: 130, bottom: 48, left: 62};
const iw = W - M.left - M.right, ih = H - M.top - M.bottom;
const ctr = root.append("div").style("font-size","13px").style("margin-bottom","6px");
ctr.append("span").html("<b>Roofline:</b> attainable TFLOP/s vs arithmetic intensity. " +
  "<span style='color:#76b900'>green = NVIDIA</span>, <span style='color:#ed1c24'>red = AMD</span>. " +
  "Hover a line for its ridge point.");

const svg = root.append("svg").attr("width",W).attr("height",H).append("g")
    .attr("transform",`translate(${M.left},${M.top})`);
const x = d3.scaleLog().domain([0.25, 4096]).range([0, iw]);
const y = d3.scaleLog().domain([0.5, 3000]).range([ih, 0]);
svg.append("g").attr("transform",`translate(0,${ih})`).call(d3.axisBottom(x).ticks(8,"~s"));
svg.append("g").call(d3.axisLeft(y).ticks(6,"~s"));
svg.append("text").attr("x",iw/2).attr("y",ih+38).attr("text-anchor","middle")
   .style("font-size","12px").text("arithmetic intensity (FLOP per byte of weights moved)");
svg.append("text").attr("transform","rotate(-90)").attr("x",-ih/2).attr("y",-44)
   .attr("text-anchor","middle").style("font-size","12px").text("attainable TFLOP/s");

// workload guide lines
svg.selectAll("wl").data(data.workloads).join("line")
   .attr("x1",d=>x(d.ai)).attr("x2",d=>x(d.ai)).attr("y1",0).attr("y2",ih)
   .attr("stroke","#cfd8dc").attr("stroke-dasharray","3 3");
svg.selectAll("wt").data(data.workloads).join("text")
   .attr("x",d=>x(d.ai)+3).attr("y",12).style("font-size","9.5px").style("fill","#78909c")
   .attr("transform",d=>`rotate(90 ${x(d.ai)+3} 12)`).text(d=>d.label);

const line = d3.line().x(d=>x(d.ai)).y(d=>y(d.perf));
const color = v => v === "NVIDIA" ? "#76b900" : "#ed1c24";
const tip = root.append("div").style("font","12px system-ui").style("height","18px")
    .style("color","#455a64");

// keep the curves inside the axes
svg.append("clipPath").attr("id","roofclip").append("rect")
   .attr("x",0).attr("y",0).attr("width",iw).attr("height",ih);
const plot = svg.append("g").attr("clip-path","url(#roofclip)");

plot.selectAll("rl").data(data.roof).join("path")
   .attr("fill","none").attr("stroke",d=>color(d.vendor)).attr("stroke-width",1.8)
   .attr("opacity",0.75).attr("d",d=>line(d.points))
   .on("mouseover", function(e,d) {
      d3.select(this).attr("stroke-width",4).attr("opacity",1);
      tip.text(`${d.name}: ${d.peak_tf} TFLOP/s peak · ${d.bw} TB/s · ridge at ` +
               `${d.ridge.toFixed(0)} FLOP/byte (= batch ${d.ridge.toFixed(0)} in fp16)`);
   })
   .on("mouseout", function() { d3.select(this).attr("stroke-width",1.8).attr("opacity",0.75); tip.text(""); });

// label the right edge, pushing labels apart so same-peak GPUs don't overlap
const labels = data.roof.map(d => ({...d, ly: y(d.peak_tf)})).sort((a,b) => a.ly - b.ly);
const MINGAP = 11;
for (let i = 1; i < labels.length; i++)
  if (labels[i].ly - labels[i-1].ly < MINGAP) labels[i].ly = labels[i-1].ly + MINGAP;
svg.selectAll("lbl").data(labels).join("text")
   .attr("x", iw + 9).attr("y", d => d.ly + 3)
   .style("font-size","9.5px").style("fill",d=>color(d.vendor)).text(d=>d.name);
// leader lines from the roof to its (possibly nudged) label
svg.selectAll("ldr").data(labels).join("line")
   .attr("x1", iw).attr("x2", iw + 7)
   .attr("y1", d => y(d.peak_tf)).attr("y2", d => d.ly)
   .attr("stroke", d => color(d.vendor)).attr("stroke-width",0.6).attr("opacity",0.6);

// ridge markers
plot.selectAll("rp").data(data.roof).join("circle")
   .attr("cx",d=>x(d.ridge)).attr("cy",d=>y(d.peak_tf)).attr("r",3)
   .attr("fill",d=>color(d.vendor));
'''
show_d3(JS, {"roof": roof, "workloads": workloads}, height=440)

**How to read this chart — it is the most useful picture in the whole track.**

- The **diagonal** part of each line is the memory-bound regime: performance is `AI × bandwidth`.
  Two GPUs with the same bandwidth are *identical* here no matter how many FLOPS they have.
- The **flat** part is the compute roof.
- The **dot** is the ridge point where they meet.
- Every dashed vertical line is a real workload. **Where it crosses a GPU's line is the performance
  you can hope for on that GPU.**

Three conclusions fall straight out:

1. **At `decode B=1` (AI=1), every GPU on this chart is deep in the memory-bound diagonal.** An H100
   and an MI300X are *not* 3× and 4× a T4 because of their FLOPS — they're ahead purely on
   bandwidth. Buying FLOPS for single-stream latency is buying nothing.
2. **At `prefill T=2048` (AI=2048), everything is on the flat roof.** Here FLOPS is all that matters,
   and the FP8-capable cards (H100/H200/B200/MI300X/MI355X) can double it again.
3. **The interesting middle** — `decode B=64` to `B=256` — is exactly where production serving runs,
   and it's where different GPUs are bound by *different* things. That's why one config doesn't
   port.

## Part 4 · The flip point, in practice

The flip point `B*` tells you what your next optimization should be:

| Where you are | Bottleneck | What helps | What does nothing |
|---|---|---|---|
| `B ≪ B*` | memory bandwidth | quantization, speculative decoding, faster HBM | more FLOPS, bigger tensor cores |
| `B ≈ B*` | balanced | kernel quality, fusion, graph capture | either extreme |
| `B ≫ B*` | compute | FP8/FP4, better GEMM kernels, more SMs/CUs | quantizing *weights* further |

Note the asymmetry in the last row: past the flip point, **weight-only int4 stops helping decode
throughput** and can even hurt (you pay dequantization compute you no longer need to save bandwidth).
That's precisely the effect measured in [Quantized Serving Showdown](./Quantized_Serving_Showdown.ipynb), where AWQ's batch-throughput gain was smaller
than its single-stream gain. The roofline predicted it.

## Part 5 · Where each GPU actually sits for a real deployment

In [ ]:
def analyze(gpu, params_b=8.0, bytes_per_weight=2.0, batch=32, prompt=1024,
            bw_eff=0.75, flop_eff=0.60):
    '''Predict decode and prefill performance and name the binding constraint.'''
    N = params_b * 1e9
    ai_decode = 2 * batch / bytes_per_weight
    ai_prefill = 2 * prompt / bytes_per_weight
    bw = gpu["bw_tbs"] * 1e12 * bw_eff
    peak = gpu["fp16_tf"] * 1e12 * flop_eff
    # decode: tokens/s across the whole batch
    decode_mem  = bw / (N * bytes_per_weight) * batch          # memory-bound estimate
    decode_comp = peak / (2 * N)                                # compute-bound estimate
    decode_tps = min(decode_mem, decode_comp)
    prefill_tps = min(peak / (2 * N), bw / (N * bytes_per_weight) * prompt)
    return {"gpu": gpu["name"], "vendor": gpu["vendor"],
            "decode_tps": decode_tps,
            "bound": "memory" if decode_mem < decode_comp else "compute",
            "prefill_tps": prefill_tps,
            "fits_fp16": gpu["vram_gb"] >= params_b * 2 * 1.25,   # weights + ~25% for KV/activations
            "ai_decode": ai_decode}

print("8B model, fp16, batch=32, 1024-token prompts:\n")
print(f"{'GPU':<14}{'decode tok/s':>14}{'bound by':>10}{'prefill tok/s':>15}{'fits?':>7}   $/1M out tok")
print("-" * 88)
for g in GPU:
    a = analyze(g)
    cpm = (g["usd_hr"] / 3600) / max(a["decode_tps"], 1e-9) * 1e6
    print(f"{g['name']:<14}{a['decode_tps']:>14,.0f}{a['bound']:>10}{a['prefill_tps']:>15,.0f}"
          f"{('yes' if a['fits_fp16'] else 'NO'):>7}   ${cpm:>8.3f}")

print("\nEvery single card is MEMORY-bound at batch 32 - AI=32 is left of every ridge point.")
print("This is why 'we bought faster tensor cores and decode didn't improve' is such a common story.")

## Part 6 · Capacity is an architecture decision

The spec everyone under-weights is **VRAM per GPU**, because it doesn't change your speed — it
changes your *topology*, and topology changes everything downstream ([Distributed & Multi-Replica Serving](./Distributed_MultiReplica_Serving.ipynb)).

A 70B model in fp16 needs ~140 GB of weights alone:

| GPU | VRAM | 70B fp16 fits? | Minimum TP | Consequence |
|---|---|---|---|---|
| A100 80GB | 80 GB | no | **TP=4** (with KV headroom) | all-reduce per layer; 4 GPUs tied together |
| H100 80GB | 80 GB | no | **TP=4** | same |
| H200 | 141 GB | barely (no KV room) | **TP=2** | halved communication |
| **MI300X** | **192 GB** | **yes** | **TP=1** | **one GPU, zero all-reduce, replicas scale linearly** |
| **MI325X** | **256 GB** | yes, with big KV pool | **TP=1** | plus room for long contexts |
| B200 | 192 GB | yes | TP=1 | same story |

**This is AMD's most underrated serving advantage and it is not a speed argument.** Running TP=1
means:

- no per-layer all-reduce ([Distributed & Multi-Replica Serving](./Distributed_MultiReplica_Serving.ipynb)'s communication tax → zero),
- no NVLink requirement,
- failure domain = one GPU instead of four,
- scaling is embarrassingly parallel replicas, which is the easy kind.

Let's quantify what avoiding TP is worth, using [Distributed & Multi-Replica Serving](./Distributed_MultiReplica_Serving.ipynb)'s model:

In [ ]:
def tp_speedup(n, comm_frac):
    return 1.0 / (1.0 / n + comm_frac * (n - 1) / n)

MODEL_B = 70
print(f"Serving a {MODEL_B}B model in fp16 (~{MODEL_B*2} GB of weights + KV):\n")
print(f"{'GPU':<14}{'VRAM':>7}{'min TP':>8}{'TP efficiency':>15}{'GPUs per replica':>18}")
print("-" * 66)
for g in GPU:
    need = MODEL_B * 2 * 1.25                       # weights + KV/activation headroom
    tp = 1
    while g["vram_gb"] * tp < need and tp < 16:
        tp *= 2
    eff = tp_speedup(tp, 0.06) / tp if tp > 1 else 1.0
    print(f"{g['name']:<14}{g['vram_gb']:>5}GB{tp:>8}{eff:>14.0%}{tp:>18}")

print(f"\nAt TP=4 you keep {tp_speedup(4, 0.06)/4:.0%} of the GPUs you paid for over NVLink,")
print(f"and only {tp_speedup(4, 0.22)/4:.0%} over PCIe; at TP=1 you keep 100% on any interconnect.")
print("A GPU with 2x the VRAM can therefore beat a 'faster' GPU on $/token even at equal bandwidth,")
print("purely by letting you avoid tensor parallelism. Capacity IS a performance feature.")

print("\nAnd the same logic for KV capacity (serving-fundamentals's formula), 70B GQA, 8k context:")
kv_per_tok = 2 * 80 * 8 * 128 * 2                    # layers, kv_heads, head_dim, bytes
for g in (x for x in GPU if x["vram_gb"] >= 80):
    free = g["vram_gb"] - MODEL_B * 2 / max(1, math.ceil(MODEL_B * 2 * 1.25 / g["vram_gb"]))
    seqs = max(0, free * 1e9 // (kv_per_tok * 8192))
    print(f"  {g['name']:<14}{free:>6.0f} GB free for KV -> ~{int(seqs):>4} concurrent 8k conversations")

## Part 7 · Measure YOUR roofline (NVIDIA or AMD)

Peak numbers are marketing. What you can actually reach is 60–85% of them, and the gap is where
kernel maturity, clocks, thermals, and your specific shapes live.

The cell below is **vendor-portable**: PyTorch's `torch.cuda` API works on ROCm too (AMD builds
alias the same namespace), so the same code measures a T4, an H100, or an MI300X. It detects which
you have, measures achieved bandwidth and GEMM throughput, and computes your **real** ridge point.

In [ ]:
# Runs on NVIDIA (CUDA) or AMD (ROCm). No vendor-specific code paths needed for this level.
# GPU-gated: degrade gracefully when PyTorch is absent, not just when the GPU is.
try:
    import torch
    HAS_GPU = torch.cuda.is_available()
except ImportError:
    torch = None
    HAS_GPU = False
    print("PyTorch is not installed here - skipping the GPU section.")
import time

def detect():
    if not HAS_GPU:
        return None
    hip = getattr(torch.version, "hip", None)
    return {"vendor": "AMD (ROCm)" if hip else "NVIDIA (CUDA)",
            "runtime": hip or torch.version.cuda,
            "name": torch.cuda.get_device_name(0),
            "vram_gb": torch.cuda.get_device_properties(0).total_memory / 1e9,
            "torch": torch.__version__}

info = detect()
if info is None:
    print("No GPU detected - Parts 1-6 already gave you the full model.")
    print("Run this cell on any CUDA or ROCm machine to measure its real roofline.")
else:
    print(f"vendor : {info['vendor']}   runtime {info['runtime']}")
    print(f"device : {info['name']}  ({info['vram_gb']:.0f} GB)   torch {info['torch']}\n")

    dev = "cuda"
    dtype = torch.float16          # universally supported; bf16 needs Ampere+/CDNA
    sync = torch.cuda.synchronize

    # --- 1. achieved memory bandwidth: a big read-write stream ---
    n = 256 * 1024 * 1024 // 2                       # 256 MB of fp16
    a = torch.empty(n, dtype=dtype, device=dev).normal_()
    b = torch.empty_like(a)
    for _ in range(3): b.copy_(a)
    sync(); t0 = time.perf_counter()
    ITERS = 30
    for _ in range(ITERS): b.copy_(a)
    sync()
    dt = time.perf_counter() - t0
    bytes_moved = 2 * a.numel() * a.element_size() * ITERS      # read + write
    achieved_bw = bytes_moved / dt / 1e12
    print(f"achieved copy bandwidth : {achieved_bw:6.2f} TB/s")

    # --- 2. achieved GEMM throughput ---
    def gemm_tflops(m, k, nn, iters=20):
        x = torch.randn(m, k, device=dev, dtype=dtype)
        w = torch.randn(k, nn, device=dev, dtype=dtype)
        for _ in range(5): torch.mm(x, w)
        sync(); t0 = time.perf_counter()
        for _ in range(iters): torch.mm(x, w)
        sync()
        return 2 * m * k * nn * iters / (time.perf_counter() - t0) / 1e12

    big = gemm_tflops(4096, 4096, 4096)
    print(f"achieved fp16 GEMM      : {big:6.1f} TFLOP/s  (4096³)")

    ridge = big * 1e12 / (achieved_bw * 1e12)
    print(f"\nYOUR measured ridge point: {ridge:.0f} FLOP/byte")
    print(f"  => in fp16, decode is memory-bound below batch ~{ridge:.0f}")
    print(f"  => in int4, that flip point drops to batch ~{ridge/4:.0f}")

    # --- 3. the shape that actually matters for decode: a skinny GEMM (batch x hidden) ---
    print("\nDecode-shaped GEMMs (this is what a real decode step looks like):")
    print(f"{'batch':>7}{'TFLOP/s':>10}{'% of big-GEMM peak':>21}")
    for bsz in (1, 8, 32, 128, 512):
        tf = gemm_tflops(bsz, 4096, 4096, iters=50)
        print(f"{bsz:>7}{tf:>10.1f}{tf/big:>20.0%}")
    print("\nSee how far below peak small batches are? That is the memory-bound regime,")
    print("measured on YOUR hardware. The roofline is not a metaphor.")

### What to do with your measured numbers

1. **Achieved bandwidth ÷ peak** tells you how healthy your memory subsystem is (expect 70–90% on a
   simple copy). Much lower → check ECC, clocks, or that you aren't sharing the GPU.
2. **Achieved GEMM ÷ peak** is your kernel-quality score for that dtype and shape. This is where
   vendor ecosystem maturity shows up most ([Portable Kernels & Precision](./Portable_Kernels_Precision_Matrix.ipynb)).
3. **Your ridge point** is the number to compare against your production batch size. If your batch
   is well below it — and it almost certainly is — then **bandwidth-side optimizations are the only
   ones that will move your decode latency.**

## Recap — the mental model that ports across vendors

| Question | Answer |
|---|---|
| Why is decode slow? | AI ≈ B, far left of every ridge point → memory-bound |
| Will quantization help? | Yes below `B*`; diminishing above it; divides `B*` by 4 |
| Will faster tensor cores help decode? | Only if you're past `B*` — usually you aren't |
| Will they help prefill? | Yes, always — prefill is at AI ≈ 2T |
| Why does my config not port to a new GPU? | Different ridge point → different bottleneck at the same batch size |
| Why care about VRAM? | It sets your TP width, and TP width taxes every layer ([Distributed & Multi-Replica Serving](./Distributed_MultiReplica_Serving.ipynb)) |
| NVIDIA vs AMD, in one line | Same physics, different vocabulary and different per-feature ecosystem maturity — see [Portable Kernels & Precision](./Portable_Kernels_Precision_Matrix.ipynb) |

### Further reading
- [Roofline: An Insightful Visual Performance Model](https://dl.acm.org/doi/10.1145/1498765.1498785) (Williams, Waterman, Patterson) — the original
- [AMD CDNA3 architecture whitepaper](https://www.amd.com/en/technologies/cdna.html) · [NVIDIA Hopper architecture whitepaper](https://resources.nvidia.com/en-us-tensor-core)
- [Transformer Inference Arithmetic](https://kipp.ly/transformer-inference-arithmetic/) — the same math, worked differently
- Measured counterparts in this repo: notebooks [Serving Fundamentals](./Serving_Fundamentals_KV_Cache_Batching.ipynb) (bandwidth-bound decode) and [Quantized Serving Showdown](./Quantized_Serving_Showdown.ipynb) (quantization)